In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, StructType, StructField, IntegerType, TimestampType, FloatType, DoubleType, LongType
from pyspark.sql.functions import trim, col , current_date, row_number, count
from pyspark.sql.window import Window
from pyspark.sql.functions import col, from_json, md5, concat_ws, lit
from delta.tables import DeltaTable

In [0]:
crypto_schema = StructType([
    StructField("id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("name", StringType(), True),
    StructField("current_price", DoubleType(), True),
    StructField("market_cap", LongType(), True),
    StructField("market_cap_rank", IntegerType(), True),
    StructField("total_volume", DoubleType(), True),
    StructField("price_change_percentage_24h", DoubleType(), True)
    
])

In [0]:

df_bronze=spark.table("crypto.bronze.brz_crypto").filter(col("ingestion_date") == current_date())

display(df_bronze.limit(10))



Parse JSON

In [0]:
from pyspark.sql.functions import from_json, col

df = df_bronze.withColumn(
    "json_data",
    from_json(col("raw_payload"), crypto_schema)
)


In [0]:
#display(df.limit(10))


In [0]:

silver_df = df.select(
    col("json_data.id").alias("coin_id"),      # rename for consistency
    col("json_data.symbol"),
    col("json_data.name"),
    col("json_data.current_price"),
    col("json_data.market_cap"),
    col("json_data.total_volume"),
    col("json_data.price_change_percentage_24h"),
    col("json_data.market_cap_rank"),
    col("_ingested_at").alias("valid_from"), # for SCD2
    col("ingestion_date"),
    col("_source")
    ).withColumn("_row_hash", md5(concat_ws("||",
        col("current_price").cast("string"),
        col("market_cap").cast("string"),
        col("total_volume").cast("string"),
        col("market_cap_rank").cast("string")
    )))


display(silver_df.limit(10))

In [0]:
 print(f"Row count: {silver_df.count()}")
 print(f"Column count: {len(silver_df.columns)}")

In [0]:
#silver_df_dedup.printSchema()


In [0]:

SILVER_TABLE = "crypto.silver.slv_crypto"

#add SCD2 columns
silver_df = silver_df \
    .withColumn("valid_to",   lit("9999-12-31").cast("date")) \
    .withColumn("is_current", lit(True))


In [0]:
#spark.sql("DROP TABLE IF EXISTS crypto.silver.slv_crypto")

In [0]:

# Check if table already exists
table_exists = spark.catalog.tableExists(SILVER_TABLE)
print(f"Table exists: {table_exists}")

#  INITIAL LOAD 
if not table_exists:
    # 1st time — write all the records
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(SILVER_TABLE)
    print("Initial load complete")

else:
    # Next times — incremental load with hash comparison
    # Step1: current records από Silver
    silver_current_df = spark.table(SILVER_TABLE) \
        .filter(col("is_current") == True) \
        .select("coin_id", "_row_hash")

    # Step2: what changed
    changes_df = silver_df.alias("new").join(
        silver_current_df.alias("old"),
        on="coin_id",
        how="left"
    ).filter(
        col("old.coin_id").isNull() |
        (col("new._row_hash") != col("old._row_hash"))
    ).select("new.*")

    # Deduplication changes_df — keep one row per coin_id (latest)
    window_changes = Window.partitionBy("coin_id").orderBy(col("valid_from").desc())

    changes_df_deduped = changes_df \
       .withColumn("rn", row_number().over(window_changes)) \
       .filter(col("rn") == 1) \
       .drop("rn")

    print("After dedup:", changes_df_deduped.count())
    print("Distinct:", changes_df_deduped.select("coin_id").distinct().count())

    # Step3: Close old records
    silver_table = DeltaTable.forName(spark, SILVER_TABLE)
    silver_table.alias("silver").merge(
        changes_df_deduped.alias("new"),
        "silver.coin_id = new.coin_id AND silver.is_current = true"
    ).whenMatchedUpdate(set={
        "is_current": lit(False),
        "valid_to":   col("new.valid_from")
    }).execute()

    # add new records
    changes_df_deduped.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(SILVER_TABLE)

    print(f"Incremental load complete — {changes_df_deduped.count()} records updated")

  
    

In [0]:
# all the records (historical + current)
spark.table("crypto.silver.slv_crypto").display()

In [0]:
# only current records
spark.table("crypto.silver.slv_crypto") \
    .filter(col("is_current") == True) \
    .display()

In [0]:
%sql
SELECT * FROM crypto.silver.slv_crypto
LIMIT 10